### Incorporating Valid County Mapping

##### Steps in the County Mapping Process
###### 1. Standardizing County Codes : The first step involves ensuring that county codes follow a consistent format. County values are converted into a standard 5-digit format and replaced missing values with zeroes and converted them into strings with appropriate padding. Additionally, any missing county entries are restored to None, ensuring that they are handled properly in subsequent processing.

###### 2. Incorporating ZCTA to County Mapping : The zcta2county crosswalk dataset is loaded to create a structured mapping between ZCTAs (ZIP Code Tabulation Areas) and their corresponding counties. A mapping is then created where each ZCTA is linked to a set of counties, allowing for easy validation against beneficiary data.

###### 3. Validating County Mappings : The dataset undergoes a validation process where the county values for each beneficiary are checked against the expected county values from the zcta2county mapping. An auxiliary dataset is created to facilitate this comparison, ensuring that each ZCTA-county pair in the dataset has a corresponding valid entry in the crosswalk. A new column is generated to flag whether a given county mapping is valid. If a county is not found in the expected set of counties for a given ZCTA, it is marked as invalid. A summary is then created, showing the total number of invalid mappings. 

###### 4. Handling Missing County Crosswalk Entries : Some ZCTAs may not have an associated county in the crosswalk dataset. To account for these cases, the process identifies all such ZCTAs and logs their occurrences. 

###### 5. Assigning County with Highest Population Percentage : For beneficiaries whose ZCTA-to-county mapping is invalid or the county is None - the county with the highest population percentage for the given ZCTA is selected as the most likely county assignment. This is done by sorting the zcta2county dataset based on population percentage and retaining the most relevant county for each ZCTA.Beneficiaries with invalid county assignments are identified, and their county values are updated based on this optimized mapping. 

In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
conn = duckdb.connect()

### total num of bene and n dead in a year

In [ ]:
years = list(range(2000, 2015))  
results = []

for year in years:
    query = f"""
    SELECT 
        {year} AS year, 
        total_bene, 
        total_deaths
    FROM (
        SELECT COUNT(*) AS total_bene
        FROM '../data/output/mbsf_mortality_denom/denom_{year}.parquet'
    ) AS bene_count,
    (
        SELECT COUNT(*) AS total_deaths
        FROM '../data/output/mbsf_mortality_denom/denom_{year}.parquet'
        WHERE EXTRACT(YEAR FROM dod) = {year}
    ) AS death_count;
    """
    
    df_year = conn.execute(query).fetchdf()
    results.append(df_year)

df_final = pd.concat(results, ignore_index=True)
df_final

In [ ]:
## query to get in table format the total of bene and deaths per year per geoboundaries
years = list(range(2000, 2015))  
results = []

for year in years:
    query = f"""
    SELECT 
        {year} AS year, 
        sum(n_bene) AS total_bene, 
        sum(n_dead) AS total_deaths
    FROM '../data/output/mbsf_mortality_denom/county_yearly/counts_{year}.parquet'
    """
    
    df_year = conn.execute(query).fetchdf()
    results.append(df_year)

df_final = pd.concat(results, ignore_index=True)
df_final

#### Plot to verify if the total of bene match across denom, zcta and county datatsets

In [ ]:
years = list(range(2000, 2015))
results_zcta = []
results_county = []
results_denom = []

for year in years:
    # Query for ZCTA-level aggregation
    query_zcta = f"""
    SELECT 
        {year} AS year, 
        SUM(n_bene) AS total_bene, 
        SUM(n_dead) AS total_deaths
    FROM '../data/output/mbsf_mortality_denom/zcta_yearly/counts_{year}.parquet'
    """
    df_zcta = conn.execute(query_zcta).fetchdf()
    results_zcta.append(df_zcta)

    # Query for County-level aggregation
    query_county = f"""
    SELECT 
        {year} AS year, 
        SUM(n_bene) AS total_bene,  
        SUM(n_dead) AS total_deaths
    FROM '../data/output/mbsf_mortality_denom/county_yearly/counts_{year}.parquet'
    """
    df_county = conn.execute(query_county).fetchdf()
    results_county.append(df_county)

      # Query for total deaths & denominator count
    query_denom = f"""
    SELECT 
        {year} AS year, 
        count(*) AS total_bene, 
        SUM(CASE WHEN EXTRACT(YEAR FROM dod) = {year} THEN 1 ELSE 0 END) AS total_deaths
    FROM '../data/output/mbsf_mortality_denom/denom_{year}.parquet'
    """
    df_denom = conn.execute(query_denom).fetchdf()
    results_denom.append(df_denom)

# Combine data across years
df_final_zcta = pd.concat(results_zcta, ignore_index=True)
df_final_county = pd.concat(results_county, ignore_index=True)
df_final_denom = pd.concat(results_denom, ignore_index=True)

# Plot deaths over time
plt.figure(figsize=(10, 5))
plt.plot(df_final_zcta['year'], df_final_zcta['total_deaths'], marker='o', linestyle='-', label="ZCTA Deaths")
plt.plot(df_final_county['year'], df_final_county['total_deaths'], marker='s', linestyle='-', label="County Deaths")
plt.plot(df_final_denom['year'], df_final_denom['total_deaths'], marker='^', linestyle='-', label="Denominator Deaths")
plt.title("Total Deaths Over Time")
plt.xlabel("Year")
plt.ylabel("Total Deaths")
plt.legend()
plt.grid()
plt.show()

# Plot total beneficiaries over time
plt.figure(figsize=(10, 5))
plt.plot(df_final_zcta['year'], df_final_zcta['total_bene'], marker='o', linestyle='--', label="ZCTA Beneficiaries")
plt.plot(df_final_county['year'], df_final_county['total_bene'], marker='s', linestyle='--', label="County Beneficiaries")
plt.plot(df_final_denom['year'], df_final_denom['total_bene'], marker='^', linestyle='--', label="Denominator Beneficiaries")
plt.title("Total Beneficiaries Over Time")
plt.xlabel("Year")
plt.ylabel("Total Beneficiaries")
plt.legend()
plt.grid()
plt.show()


##### Query to see the first five rows of the zcta and county counts and strat counts datasets at daily and yearlry level

In [ ]:
### at zcta level 
file_paths = {
    "counts_2008": "../data/output/mbsf_mortality_denom/zcta_yearly/counts_2008.parquet",
    "strat_counts_2008": "../data/output/mbsf_mortality_denom/zcta_yearly/strat_counts_2008.parquet",
    "strat_sparse_counts_2008": "../data/output/mbsf_mortality_denom/zcta_daily/strat_sparse_counts_2008.parquet",
    "counts_daily_2008": "../data/output/mbsf_mortality_denom/zcta_daily/counts_2008.parquet",
}

dataframes = {
    name: duckdb.execute(f"SELECT * FROM '{path}' LIMIT 5").fetchdf()
    for name, path in file_paths.items()
}

for name, df in dataframes.items():
    print(f"\nHeaders for {name}:")
    print(df.head())


In [ ]:
### at county level 
file_paths = {
    "counts_2008": "../data/output/mbsf_mortality_denom/county_yearly/counts_2008.parquet",
    "strat_counts_2008": "../data/output/mbsf_mortality_denom/county_yearly/strat_counts_2008.parquet",
    "strat_sparse_counts_2008": "../data/output/mbsf_mortality_denom/county_daily/strat_sparse_counts_2008.parquet",
    "counts_daily_2008": "../data/output/mbsf_mortality_denom/county_daily/counts_2008.parquet",
}

dataframes = {
    name: duckdb.execute(f"SELECT * FROM '{path}' LIMIT 5").fetchdf()
    for name, path in file_paths.items()
}

for name, df in dataframes.items():
    print(f"\nHeaders for {name}:")
    print(df.head())

##### QC based on a dataset for the year 2008

In [ ]:
## num of unique zcta and county in the denom dataset
conn.execute(f"""
        SELECT count(distinct zcta) AS zcta, count(distinct county) AS county
        FROM '../data/output/mbsf_mortality_denom/denom_2008.parquet'
        """).fetchdf()


In [ ]:
## num of unique zcta and county in the crosswalk
conn.execute(f"""
        SELECT count(distinct zcta) as zcta, count(distinct county) AS county
        FROM '../data/input/us_geoboundaries__census/us_xwalks__census/zcta2county_yearly/us_xwalks__census__zcta2county_yearly__2008.parquet'
        """).fetchdf()

In [ ]:
## whether any county has a None value 
conn.execute(f"""
        SELECT distinct(zcta), county
        FROM '../data/output/mbsf_mortality_denom/denom_2008.parquet'
        where county is null
        """).fetchdf()

In [ ]:
## whether any county has a None value 
conn.execute(f"""
        SELECT distinct d.zcta, d.county
        FROM '../data/output/mbsf_mortality_denom/denom_2008.parquet' as d
        LEFT JOIN '../data/input/us_geoboundaries__census/us_xwalks__census/zcta2county_yearly/us_xwalks__census__zcta2county_yearly__2008.parquet' AS c
        ON d.zcta = c.zcta
        WHERE c.zcta IS NULL
        """).fetchdf()

In [ ]:
## check if these zcta's are present in the zcta2county dataset 
conn.execute(f"""
    SELECT *
    FROM '../data/input/us_geoboundaries__census/us_xwalks__census/zcta2county_yearly/us_xwalks__census__zcta2county_yearly__2008.parquet'
    WHERE zcta IN ('96929','96950','96917','96913','96799','96932')
    """).fetchdf()

##### seems like the zctas are not present in the zcta2county crosswalk ; hence the counties are not replaced

#### Detailed exploration of the zcta mismatch bettern the denom dataset and the zcta2county crosswalk

In [ ]:
#mismatch zctas between zcta2county file and denom file 
denom = '../data/output/mbsf_mortality_denom/denom_{year}.parquet'  
zcta2county = '../data/input/us_geoboundaries__census/us_xwalks__census/zcta2county_yearly/us_xwalks__census__zcta2county_yearly__{year}.parquet'
years = list(range(2000, 2002))  
results = []

for year in years:
    denom = denom.format(year=year)
    zcta2county = zcta2county.format(year=year)
    conn.execute(f"CREATE OR REPLACE TABLE denom AS SELECT DISTINCT zcta FROM '{denom}'")
    conn.execute(f"CREATE OR REPLACE TABLE zcta2county AS SELECT DISTINCT zcta FROM '{zcta2county}'")

    # Find ZCTAs in file1 but not in file2
    zcta_only_in_denom = conn.execute("""
        SELECT zcta FROM denom 
        WHERE zcta NOT IN (SELECT zcta FROM zcta2county)
    """).df()
    zcta_only_in_denom["year"] = year
    zcta_only_in_denom["status"] = "Only in denom file"

    # Find ZCTAs in file2 but not in file1
    zcta_only_in_zcta2county = conn.execute("""
        SELECT zcta FROM zcta2county 
        WHERE zcta NOT IN (SELECT zcta FROM denom)
    """).df()
    zcta_only_in_zcta2county["year"] = year
    zcta_only_in_zcta2county["status"] = "Only in zcta2county"

    # Append results
    results.append(zcta_only_in_denom)
    results.append(zcta_only_in_zcta2county)

# Combine results 
df_zcta_differences = pd.concat(results, ignore_index=True)
df_zcta_differences

In [ ]:
##just the counts 
denom_path = '../data/output/mbsf_mortality_denom/denom_{year}.parquet'
zcta2county = '../data/input/us_geoboundaries__census/us_xwalks__census/zcta2county_yearly/us_xwalks__census__zcta2county_yearly__{year}.parquet'
years = range(2000, 2015)
mismatch_counts = [] 

for year in years:
    denom_file = denom_path.format(year=year)
    zcta2county_file = zcta2county.format(year=year)

    conn.execute(f"CREATE OR REPLACE TABLE denom AS SELECT DISTINCT zcta FROM '{denom_file}'")
    conn.execute(f"CREATE OR REPLACE TABLE zcta2county AS SELECT DISTINCT zcta FROM '{zcta2county_file}'")

    # counts of mismatches
    count_denom = conn.execute("SELECT COUNT(*) FROM denom WHERE zcta NOT IN (SELECT zcta FROM zcta2county)").fetchone()[0]
    count_zcta2county = conn.execute("SELECT COUNT(*) FROM zcta2county WHERE zcta NOT IN (SELECT zcta FROM denom)").fetchone()[0]

    mismatch_counts.append({"year": year, "only_in_denom": count_denom, "only_in_zcta2county": count_zcta2county})
    df_counts = pd.DataFrame(mismatch_counts) 

print(df_counts)

#### QC of the county counts dataset for the year 2008 

In [ ]:
conn.execute(f"""
        SELECT count(distinct county)
        FROM '../data/output/mbsf_mortality_denom/county_yearly/counts_2008.parquet'
        """).fetchdf()

In [ ]:
conn.execute(f"""
        SELECT sum(n_bene), sum(n_dead)
        FROM '../data/output/mbsf_mortality_denom/county_yearly/counts_2008.parquet'
        """).fetchdf()

In [ ]:
conn.execute(f"""
        SELECT count(*) as total_bene, 
               SUM(CASE WHEN EXTRACT(YEAR FROM dod) = 2008 THEN 1 ELSE 0 END) AS deceased_bene
        FROM '../data/output/mbsf_mortality_denom/denom_2008.parquet'
        WHERE zcta IN ('96929','96950','96917','96913','96799','96932')
        """).fetchdf()

##### From the above two queries we see that the difference in the total num of dead is becuase of the missing county in the denom file (1815571 - 1815568) = 3 and total bene (39471539 - 39471522) = 17